CNN_Scratch

In [ ]:
# ======================================================
# CNN SCRATCH - TRAINING & SAVE (STREAMLIT READY)
# ======================================================

import os
import time
import joblib
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# ======================================================
# DEVICE
# ======================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", device)

# ======================================================
# PATH CONFIG
# ======================================================
BASE_DIR = "processed_dataset"

TRAIN_DIR = os.path.join(BASE_DIR, "train")
VAL_DIR   = os.path.join(BASE_DIR, "val")

# 📌 SIMPAN KE: Dashboard/src/CNN/model
SAVE_DIR = os.path.join(
    os.path.dirname(__file__),
    "model"
)
os.makedirs(SAVE_DIR, exist_ok=True)

MODEL_PATH = os.path.join(SAVE_DIR, "cnn_scratch_best.pkl")

# ======================================================
# CONFIG
# ======================================================
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 4
EPOCHS = 30
PATIENCE = 5
LR = 1e-3

# ======================================================
# TRANSFORM
# ======================================================
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ======================================================
# DATASET & DATALOADER
# ======================================================
train_ds = datasets.ImageFolder(TRAIN_DIR, transform=transform)
val_ds   = datasets.ImageFolder(VAL_DIR, transform=transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

class_names = train_ds.classes
print("Classes:", class_names)

# ======================================================
# CNN MODEL
# ======================================================
class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

model = SimpleCNN(NUM_CLASSES).to(device)
print(model)

# ======================================================
# LOSS & OPTIMIZER
# ======================================================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

# ======================================================
# TRAINING LOOP
# ======================================================
best_val_acc = 0.0
patience_counter = 0

train_losses, val_losses = [], []
train_accs, val_accs = [], []

start_time = time.time()

for epoch in range(EPOCHS):
    # ---------------- TRAIN ----------------
    model.train()
    correct, total, running_loss = 0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, preds = outputs.max(1)
        total += labels.size(0)
        correct += preds.eq(labels).sum().item()

    train_acc = correct / total

    # ---------------- VALIDATION ----------------
    model.eval()
    val_correct, val_total, val_loss = 0, 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, preds = outputs.max(1)
            val_total += labels.size(0)
            val_correct += preds.eq(labels).sum().item()

    val_acc = val_correct / val_total

    train_losses.append(running_loss / len(train_loader))
    val_losses.append(val_loss / len(val_loader))
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(
        f"[Epoch {epoch+1}/{EPOCHS}] "
        f"Train Acc: {train_acc*100:.2f}% | "
        f"Val Acc: {val_acc*100:.2f}%"
    )

    # ---------------- SAVE BEST ----------------
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0

        artifact = {
            "model_state_dict": model.state_dict(),
            "architecture": "CNN Scratch",
            "num_classes": NUM_CLASSES,
            "class_names": class_names,
            "img_size": IMG_SIZE,
            "normalization": {
                "mean": [0.485, 0.456, 0.406],
                "std": [0.229, 0.224, 0.225]
            },
            "best_val_acc": best_val_acc,
            "epoch": epoch + 1
        }

        joblib.dump(artifact, MODEL_PATH)
        print(f">>> Best model saved: {MODEL_PATH}")

    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(">>> Early stopping triggered!")
            break

total_time = time.time() - start_time
print(f"\nTraining finished in {total_time:.2f} seconds")
print("Best Val Acc:", best_val_acc)


Resnet50_LoRA_Fine-Tuning

Efficientnet_B0_Baseline

In [ ]:
# ======================================================
# EfficientNet-B0 Baseline (Training + Save for Streamlit)
# ======================================================

import os
import torch
import joblib
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
from torchvision import datasets, transforms, models
from torchvision.models import EfficientNet_B0_Weights
from torch.utils.data import DataLoader

# ======================================================
# DEVICE
# ======================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", device)

# ======================================================
# PATH CONFIG
# ======================================================
BASE_DIR = "processed_dataset"
TRAIN_DIR = os.path.join(BASE_DIR, "train")
VAL_DIR   = os.path.join(BASE_DIR, "test")  # dipakai sebagai validation

SAVE_DIR = os.path.join(
    os.path.dirname(__file__),  # Dashboard/src/CNN
    "model"
)
os.makedirs(SAVE_DIR, exist_ok=True)

MODEL_PATH = os.path.join(SAVE_DIR, "efficientnet_b0_baseline.pkl")

# ======================================================
# CONFIG
# ======================================================
NUM_CLASSES = 4
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 15
PATIENCE = 3
LR = 1e-4

# ======================================================
# TRANSFORMS
# ======================================================
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ======================================================
# DATASET & DATALOADER
# ======================================================
train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=transform)
val_dataset   = datasets.ImageFolder(VAL_DIR, transform=transform)

class_names = train_dataset.classes

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

# ======================================================
# MODEL INITIALIZATION
# ======================================================
weights = EfficientNet_B0_Weights.DEFAULT
model = models.efficientnet_b0(weights=weights)

in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, NUM_CLASSES)

model = model.to(device)

# ======================================================
# LOSS & OPTIMIZER
# ======================================================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)
scaler = torch.cuda.amp.GradScaler()

# ======================================================
# TRAINING LOOP
# ======================================================
best_val_acc = 0.0
patience_counter = 0

train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(EPOCHS):
    # ---------------- TRAIN ----------------
    model.train()
    running_loss, correct, total = 0, 0, 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        _, preds = outputs.max(1)
        total += labels.size(0)
        correct += preds.eq(labels).sum().item()

    train_loss = running_loss / len(train_loader)
    train_acc = correct / total

    # ---------------- VALIDATION ----------------
    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            with torch.cuda.amp.autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, preds = outputs.max(1)
            val_total += labels.size(0)
            val_correct += preds.eq(labels).sum().item()

    val_loss /= len(val_loader)
    val_acc = val_correct / val_total

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(f"[Epoch {epoch+1}] Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

    # ---------------- EARLY STOP + SAVE BEST ----------------
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0

        artifact = {
            "model_state_dict": model.state_dict(),
            "architecture": "EfficientNet-B0",
            "num_classes": NUM_CLASSES,
            "class_names": class_names,
            "img_size": IMG_SIZE,
            "normalization": {
                "mean": [0.485, 0.456, 0.406],
                "std": [0.229, 0.224, 0.225]
            }
        }

        joblib.dump(artifact, MODEL_PATH)
        print(f">>> Best model saved: {MODEL_PATH}")

    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print("Early stopping triggered.")
            break

print("\n[SUCCESS] EfficientNet-B0 baseline model ready for Streamlit")
print("Saved at:", MODEL_PATH)
